# Trajectory Data from the Simulation

In [ ]:
import os
import sys

_NGM_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if _NGM_ROOT not in sys.path:
    sys.path.insert(0, _NGM_ROOT)
from ngm_paths import simulation_results_dir

RESULTS_DIR = simulation_results_dir()
CASE_RUNS_DIR = os.path.join(RESULTS_DIR, "case_runs")

# ==========================================================================
# CASE STUDY 1 — Freeway, 100% CAV (homogeneous) traffic
# Goal: quantify how V2V communication quality (latency, packet loss) affects
# CAV platoon car-following stability. Baseline Ideal = 0 ms / 0% / 50 m range
# / 3-vehicle max lookahead. Compared against High Latency (1.0 s), High Packet
# Loss (50%), and combined High Latency + High Packet Loss.
# ==========================================================================
# CASE STUDY 2 — Market Penetration Rate (MPR) on Freeway & Arterial
# Heterogeneous mixes of SV / HV / CAV / CAHV. Arterial CSVs are currently
# placeholders (code auto-skips missing files; will run when they appear).
# ==========================================================================
# CASE STUDY 3 — Single intersection, 50/50 SV/CAV (safety via TTC)
# Compares TTC distribution between Human-driven followers (IDM / MOBIL) and
# CAV followers (C-IDM / C-MOBIL).
# ==========================================================================
# CASE STUDY 4 — Shockwave (SW_SV, SW_AV, SW_CAV single CSVs)
# Trajectory plots use dotted lines for all vehicles (no type-based linestyle).
# ==========================================================================

# Labels grouped as CS4 in summaries and special trajectory styling
CS4_LABELS = frozenset({"SW_SV", "SW_AV", "SW_CAV"})

# ---------------------------------------------------------------------------
# Which case studies to process? (trajectory, FD, TTC cells respect this)
#   "all"              -> CS1 + CS2 + CS3 + CS4
#   "CS1" / "CS2" / "CS3" / "CS4"  -> one case study only
#   ["CS1", "CS4"]    -> any list/tuple of case-study ids
# ---------------------------------------------------------------------------
CASE_STUDIES_TO_RUN = "CS1"  # <-- change this, then re-run this cell

_ALL_CASE_IDS = ("CS1", "CS2", "CS3", "CS4")


def _normalize_case_studies_to_run(selection):
    if selection is None or selection == "all":
        return set(_ALL_CASE_IDS)
    if isinstance(selection, str):
        key = selection.strip().upper()
        if key == "ALL":
            return set(_ALL_CASE_IDS)
        if key not in _ALL_CASE_IDS:
            raise ValueError(
                f"Unknown case study {selection!r}; use 'all' or one of {_ALL_CASE_IDS}"
            )
        return {key}
    return {str(c).strip().upper() for c in selection}


ACTIVE_CASE_STUDIES = _normalize_case_studies_to_run(CASE_STUDIES_TO_RUN)


def iter_scenario_labels():
    """Scenario labels in FILE_META order, filtered by ACTIVE_CASE_STUDIES."""
    for label, meta in FILE_META.items():
        if meta.get("case") in ACTIVE_CASE_STUDIES:
            yield label


def labels_for_case(case_id):
    """All enabled scenario labels for one case study (e.g. 'CS3')."""
    cid = str(case_id).strip().upper()
    if cid not in ACTIVE_CASE_STUDIES:
        return []
    return [lab for lab, m in FILE_META.items() if m.get("case") == cid]


def active_case_study_ids():
    """Enabled case-study ids in CS1..CS4 order."""
    return tuple(c for c in _ALL_CASE_IDS if c in ACTIVE_CASE_STUDIES)


# Optional fallbacks: single CSV per scenario when case_runs/<Label>/ has no run_*.csv
FILES_LEGACY = {
    # --- Case Study 1: V2V communication quality (10–16) ---
    "CS1_HDV_Baseline":         rf"{RESULTS_DIR}\10.csv",
    "CS1_Ideal":                rf"{RESULTS_DIR}\11.csv",
    "CS1_HighLoss":             rf"{RESULTS_DIR}\12.csv",
    "CS1_MidLatency_LowLoss":   rf"{RESULTS_DIR}\13.csv",
    "CS1_MidLatency_MidLoss":   rf"{RESULTS_DIR}\14.csv",
    "CS1_HighLatency":          rf"{RESULTS_DIR}\15.csv",
    "CS1_HighLatency_HighLoss": rf"{RESULTS_DIR}\16.csv",
    # --- Case Study 2: MPR on Freeway (21–29) ---
    "CS2_Fwy_100SV":                  rf"{RESULTS_DIR}\21.csv",
    "CS2_Fwy_90SV_10HV":              rf"{RESULTS_DIR}\22.csv",
    "CS2_Fwy_80SV_20HV":              rf"{RESULTS_DIR}\23.csv",
    "CS2_Fwy_45CAV_5CAHV_45SV_5HV": rf"{RESULTS_DIR}\24.csv",
    "CS2_Fwy_100CAV":                 rf"{RESULTS_DIR}\25.csv",
    "CS2_Fwy_90CAV_10CAHV":           rf"{RESULTS_DIR}\26.csv",
    "CS2_Fwy_80CAV_20CAHV":           rf"{RESULTS_DIR}\27.csv",
    "CS2_Fwy_90SV_10CAHV":            rf"{RESULTS_DIR}\28.csv",
    "CS2_Fwy_90CAV_10HV":             rf"{RESULTS_DIR}\29.csv",
    # --- Case Study 2: MPR on Arterial (30–33) ---
    "CS2_Art_90SV_10HV":              rf"{RESULTS_DIR}\30.csv",
    "CS2_Art_80SV_20HV":              rf"{RESULTS_DIR}\31.csv",
    "CS2_Art_90CAV_10CAHV":           rf"{RESULTS_DIR}\32.csv",
    "CS2_Art_80CAV_20CAHV":           rf"{RESULTS_DIR}\33.csv",
    # --- Case Study 3 ---
    "CS3_50SV_50CAV":                 rf"{RESULTS_DIR}\34.csv",
    "CS3_50SV_50AV":                  rf"{RESULTS_DIR}\35.csv",
    # --- Case Study 4: Shockwave (single-file CSVs) ---
    "SW_SV":  os.path.join(CASE_RUNS_DIR, "SW_SV.csv"),
    "SW_AV":  os.path.join(CASE_RUNS_DIR, "SW_AV.csv"),
    "SW_CAV": os.path.join(CASE_RUNS_DIR, "SW_CAV.csv"),
}


def discover_run_csv_paths(case_label: str):
    """Return sorted CSV paths: case_runs/<case_label>/run_*.csv, else legacy single file if present."""
    folder = os.path.join(CASE_RUNS_DIR, case_label)
    if os.path.isdir(folder):
        names = sorted(
            fn
            for fn in os.listdir(folder)
            if fn.lower().endswith(".csv")
            and (fn.startswith("run_") or fn.startswith("run-"))
        )
        if names:
            return [os.path.join(folder, fn) for fn in names]
    legacy = FILES_LEGACY.get(case_label)
    if legacy and os.path.isfile(legacy):
        return [legacy]
    return []

# Metadata used by downstream summary + TTC cells so the tables group cleanly.
# freeway_type mirrors GUI Geometry["Freeway_Type"] (CS1=on_off, CS2 freeway=on_off_on_off).
# FD study segment uses env + freeway_type in the Flow-Density cell (`fd_bounds_for_label`).
FILE_META = {
    "CS1_HDV_Baseline":               {"case": "CS1", "env": "Freeway",  "scenario": "100% SV (HDV baseline, PT + DDM)", "freeway_type": "on_off"},
    "CS1_Ideal":                      {"case": "CS1", "env": "Freeway",  "scenario": "Ideal Connectivity (0 ms / 0 % loss)", "freeway_type": "on_off"},
    "CS1_HighLoss":                   {"case": "CS1", "env": "Freeway",  "scenario": "High Packet Loss (0 ms / 50 % loss)", "freeway_type": "on_off"},
    "CS1_MidLatency_LowLoss":         {"case": "CS1", "env": "Freeway",  "scenario": "Mid Latency + Low Packet Loss (0.5 s / 25 % loss)", "freeway_type": "on_off"},
    "CS1_MidLatency_MidLoss":         {"case": "CS1", "env": "Freeway",  "scenario": "Mid Latency + Mid Packet Loss (0.5 s / 50 % loss)", "freeway_type": "on_off"},
    "CS1_HighLatency":                {"case": "CS1", "env": "Freeway",  "scenario": "High Latency (1.0 s / 0 % loss)", "freeway_type": "on_off"},
    "CS1_HighLatency_HighLoss":       {"case": "CS1", "env": "Freeway",  "scenario": "High Latency + High Packet Loss (1.0 s / 50 % loss)", "freeway_type": "on_off"},
    "CS2_Fwy_100SV":                  {"case": "CS2", "env": "Freeway",  "scenario": "100% SVs (no HVs)", "freeway_type": "on_off_on_off"},
    "CS2_Fwy_90SV_10HV":              {"case": "CS2", "env": "Freeway",  "scenario": "90% SVs – 10% HVs", "freeway_type": "on_off_on_off"},
    "CS2_Fwy_80SV_20HV":              {"case": "CS2", "env": "Freeway",  "scenario": "80% SVs – 20% HVs", "freeway_type": "on_off_on_off"},
    "CS2_Fwy_45CAV_5CAHV_45SV_5HV": {"case": "CS2", "env": "Freeway",  "scenario": "45% CAVs – 5% CAHVs / 45% SVs – 5% HVs", "freeway_type": "on_off_on_off"},
    "CS2_Fwy_100CAV":                 {"case": "CS2", "env": "Freeway",  "scenario": "100% CAVs (no CAHVs / no human-driven trucks)", "freeway_type": "on_off_on_off"},
    "CS2_Fwy_90CAV_10CAHV":           {"case": "CS2", "env": "Freeway",  "scenario": "90% CAVs – 10% CAHVs", "freeway_type": "on_off_on_off"},
    "CS2_Fwy_80CAV_20CAHV":           {"case": "CS2", "env": "Freeway",  "scenario": "80% CAVs – 20% CAHVs", "freeway_type": "on_off_on_off"},
    "CS2_Fwy_90SV_10CAHV":            {"case": "CS2", "env": "Freeway",  "scenario": "90% SVs – 10% CAHVs", "freeway_type": "on_off_on_off"},
    "CS2_Fwy_90CAV_10HV":             {"case": "CS2", "env": "Freeway",  "scenario": "90% CAVs – 10% HVs", "freeway_type": "on_off_on_off"},
    "CS2_Art_90SV_10HV":              {"case": "CS2", "env": "Arterial", "scenario": "90% SVs – 10% HVs"},
    "CS2_Art_80SV_20HV":              {"case": "CS2", "env": "Arterial", "scenario": "80% SVs – 20% HVs"},
    "CS2_Art_90CAV_10CAHV":           {"case": "CS2", "env": "Arterial", "scenario": "90% CAVs – 10% CAHVs"},
    "CS2_Art_80CAV_20CAHV":           {"case": "CS2", "env": "Arterial", "scenario": "80% CAVs – 20% CAHVs"},
    "CS3_50SV_50CAV":                 {"case": "CS3", "env": "Intersection", "scenario": "50% SVs – 50% CAVs"},
    "CS3_50SV_50AV":                  {"case": "CS3", "env": "Intersection", "scenario": "50% SVs – 50% AVs"},
    "SW_SV":                          {"case": "CS4", "env": "Freeway", "scenario": "Shockwave — 100% SV", "freeway_type": "on_off_on_off"},
    "SW_AV":                          {"case": "CS4", "env": "Freeway", "scenario": "Shockwave — 100% AV", "freeway_type": "on_off_on_off"},
    "SW_CAV":                         {"case": "CS4", "env": "Freeway", "scenario": "Shockwave — 100% CAV", "freeway_type": "on_off_on_off"},
}

_n_enabled = sum(1 for _ in iter_scenario_labels())
print(f"Case studies enabled: {', '.join(active_case_study_ids())} ({_n_enabled} scenario(s))")

Case studies enabled: CS1 (7 scenario(s))


# Trajectory Plots / Avg Speed and TT Calculation

In [17]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------------------------
# 1) CONFIGURATION & CONSTANTS
# ---------------------------

COMPARE_RAW_SMOOTHED = False

# Trajectory figure size (inches): width fixed; height = row height × number of lane rows
TRAJ_FIG_WIDTH_IN = 20
TRAJ_ROW_HEIGHT_IN = 3       # default (CS1–CS3)
TRAJ_ROW_HEIGHT_CS4_IN = 6   # CS4 shockwave: 2× row height

MIN_LANE_STAY = 4.0  

TYPE_LABELS = {
    "0": "SV (Small Vehicle)",
    "1": "AV (Automated Vehicle)",
    "2": "HV (Heavy Vehicle)",
    "3": "CAV (Connected & Autonomous Vehicle)",
}

# 508 Compliant Palette (Okabe-Ito) - Used as secondary distinction
TYPE_COLORS = {
    "0": (153/255, 153/255, 153/255),  # SV: Neutral Grey
    "1": (86/255,  180/255, 233/255),  # AV: Sky Blue
    "2": (230/255, 159/255, 0/255),    # HV: Orange
    "3": (0/255,   158/255, 115/255),  # CAV: Bluish Green
}

TYPE_COLORS = {"0": (0, 0, 0), "1": (0, 0, 0), "2": (0, 0, 0), "3": (0, 0, 0)}

# 508 Compliant Markers - Primary distinction (Shape instead of Color)
TYPE_MARKERS = {
    "0": "o",  # SV: Circle
    "1": "^",  # AV: Triangle Up
    "2": "s",  # HV: Square
    "3": "D",  # CAV: Diamond
}

OUT_DIR = "processed_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ---------------------------
# 2) DATA LOADING
# ---------------------------

def load_trajectory_csv(path):
    usecols = ["time", "id", "type", "v", "lane", "x", "y", "lane_pos"]
    try:
        df = pd.read_csv(path, usecols=lambda c: c in usecols)
    except ValueError:
        df = pd.read_csv(path)

    df = df.dropna(subset=["time", "id", "lane"])
    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df["id"] = pd.to_numeric(df["id"], errors="coerce")
    
    if "type" in df.columns:
        df["type"] = df["type"].astype(str).str.split('.').str[0]
    else:
        df["type"] = "0"

    if "x" in df.columns:
        df["s"] = pd.to_numeric(df["x"], errors="coerce")
    elif "lane_pos" in df.columns:
        df["s"] = pd.to_numeric(df["lane_pos"], errors="coerce")
    else:
        raise ValueError(f"No position column (x or lane_pos) found in {path}")

    df = df.dropna(subset=["s"])
    df = df.sort_values(["id", "time"]).reset_index(drop=True)
    return df

# ---------------------------
# 3) LOGICAL LANE MAPPING (must match freeway_sim.py)
# ---------------------------
# In the highway, each "logical lane" (e.g. Lane 1) is one continuous lane that spans
# many SUMO edge/lane names (Input_0, Weaving_Area_1, Output_0, etc.). We use the
# same all_lane_names as freeway_sim.py so that Lane 1 includes ALL those segments.

def _build_freeway_lane_lists():
    """Same lane-name lists as in freeway_sim.py for each freeway_type (num_lanes=6)."""
    # on_off: one ramp (0) + mainline 1..6
    on_off = [
        ["On_Ramp_0", ":Weaving_Start_0_0", "Weaving_Area_0", ":Weaving_End_0_0", "Off_Ramp_0"],
        ["Input_0", ":Weaving_Start_1_0", "Weaving_Area_1", ":Weaving_End_1_0", "Output_0"],
        ["Input_1", ":Weaving_Start_1_1", "Weaving_Area_2", ":Weaving_End_1_1", "Output_1"],
        ["Input_2", ":Weaving_Start_1_2", "Weaving_Area_3", ":Weaving_End_1_2", "Output_2"],
        ["Input_3", ":Weaving_Start_1_3", "Weaving_Area_4", ":Weaving_End_1_3", "Output_3"],
        ["Input_4", ":Weaving_Start_1_4", "Weaving_Area_5", ":Weaving_End_1_4", "Output_4"],
        ["Input_5", ":Weaving_Start_1_5", "Weaving_Area_6", ":Weaving_End_1_5", "Output_5"],
    ]
    # on_on_off_off: ramp segments (0) + mainline 1..6
    on_on_off_off = [
        ["On_Ramp1_0", ":On_Ramp1_End_0_0", "On_Ramp1_Taper_0", ":On_Ramp1_Taper_End_0_0"],
        ["On_Ramp2_0", ":Weaving_Start_0_0", "Weaving_Area_0", ":Weaving_End_0_0", "Off_Ramp1_0"],
        [":Off_Ramp2_Taper_Start_0_0", "Off_Ramp2_Taper_0", ":Off_Ramp2_Start_0_0", "Off_Ramp2_0"],
        ["Input_0", ":On_Ramp1_End_1_0", "On_Ramp1_Taper_1", ":On_Ramp1_Taper_End_0_0", "Before_Weaving_0", ":Weaving_Start_1_0", "Weaving_Area_1", ":Weaving_End_1_0", "After_Weaving_0", ":Off_Ramp2_Taper_Start_0_1", "Off_Ramp2_Taper_1", ":Off_Ramp2_Start_1_0", "Output_0"],
        ["Input_1", ":On_Ramp1_End_1_1", "On_Ramp1_Taper_2", ":On_Ramp1_Taper_End_0_1", "Before_Weaving_1", ":Weaving_Start_1_1", "Weaving_Area_2", ":Weaving_End_1_1", "After_Weaving_1", ":Off_Ramp2_Taper_Start_0_2", "Off_Ramp2_Taper_2", ":Off_Ramp2_Start_1_1", "Output_1"],
        ["Input_2", ":On_Ramp1_End_1_2", "On_Ramp1_Taper_3", ":On_Ramp1_Taper_End_0_2", "Before_Weaving_2", ":Weaving_Start_1_2", "Weaving_Area_3", ":Weaving_End_1_2", "After_Weaving_2", ":Off_Ramp2_Taper_Start_0_3", "Off_Ramp2_Taper_3", ":Off_Ramp2_Start_1_2", "Output_2"],
        ["Input_3", ":On_Ramp1_End_1_3", "On_Ramp1_Taper_4", ":On_Ramp1_Taper_End_0_3", "Before_Weaving_3", ":Weaving_Start_1_3", "Weaving_Area_4", ":Weaving_End_1_3", "After_Weaving_3", ":Off_Ramp2_Taper_Start_0_4", "Off_Ramp2_Taper_4", ":Off_Ramp2_Start_1_3", "Output_3"],
        ["Input_4", ":On_Ramp1_End_1_4", "On_Ramp1_Taper_5", ":On_Ramp1_Taper_End_0_4", "Before_Weaving_4", ":Weaving_Start_1_4", "Weaving_Area_5", ":Weaving_End_1_4", "After_Weaving_4", ":Off_Ramp2_Taper_Start_0_5", "Off_Ramp2_Taper_5", ":Off_Ramp2_Start_1_4", "Output_4"],
        ["Input_5", ":On_Ramp1_End_1_5", "On_Ramp1_Taper_6", ":On_Ramp1_Taper_End_0_5", "Before_Weaving_5", ":Weaving_Start_1_5", "Weaving_Area_6", ":Weaving_End_1_5", "After_Weaving_5", ":Off_Ramp2_Taper_Start_0_6", "Off_Ramp2_Taper_6", ":Off_Ramp2_Start_1_5", "Output_5"],
    ]
    # on_off_on_off: two ramp lanes (0,1) + mainline 2..6
    on_off_on_off = [
        ["On_Ramp1_0", ":Weaving1_Start_0_0", "Weaving_Area1_0", ":Weaving1_End_0_0", "Off_Ramp1_0"],
        ["On_Ramp2_0", ":Weaving2_Start_0_0", "Weaving_Area2_0", ":Weaving2_End_0_0", "Off_Ramp2_0"],
        ["Input_0", ":Weaving1_Start_1_0", "Weaving_Area1_1", ":Weaving1_End_1_0", "Between_Weaving_0", ":Weaving2_Start_1_0", "Weaving_Area2_1", ":Weaving2_End_1_0", "Output_0"],
        ["Input_1", ":Weaving1_Start_1_1", "Weaving_Area1_2", ":Weaving1_End_1_1", "Between_Weaving_1", ":Weaving2_Start_1_1", "Weaving_Area2_2", ":Weaving2_End_1_1", "Output_1"],
        ["Input_2", ":Weaving1_Start_1_2", "Weaving_Area1_3", ":Weaving1_End_1_2", "Between_Weaving_2", ":Weaving2_Start_1_2", "Weaving_Area2_3", ":Weaving2_End_1_2", "Output_2"],
        ["Input_3", ":Weaving1_Start_1_3", "Weaving_Area1_4", ":Weaving1_End_1_3", "Between_Weaving_3", ":Weaving2_Start_1_3", "Weaving_Area2_4", ":Weaving2_End_1_3", "Output_3"],
        ["Input_4", ":Weaving1_Start_1_4", "Weaving_Area1_5", ":Weaving1_End_1_4", "Between_Weaving_4", ":Weaving2_Start_1_4", "Weaving_Area2_5", ":Weaving2_End_1_4", "Output_4"],
        ["Input_5", ":Weaving1_Start_1_5", "Weaving_Area1_6", ":Weaving1_End_1_5", "Between_Weaving_5", ":Weaving2_Start_1_5", "Weaving_Area2_6", ":Weaving2_End_1_5", "Output_5"],
    ]
    return {"on_off": on_off, "on_on_off_off": on_on_off_off, "on_off_on_off": on_off_on_off}


def _lane_segment_to_idx_from_lists(all_lane_names):
    """Build segment -> logical_lane_index (same as freeway_sim lane_segment_to_idx)."""
    out = {}
    for lane_idx, names in enumerate(all_lane_names):
        for name in names:
            out[name] = lane_idx
    return out


def _infer_freeway_type(df):
    """Infer freeway_type from lane names present (must match geometry used in sim)."""
    lanes_str = set(df["lane"].astype(str))
    if any("Before_Weaving" in s or "On_Ramp1_Taper" in s for s in lanes_str):
        return "on_on_off_off"
    if any("Weaving_Area1_" in s or "Weaving_Area2_" in s or "Between_Weaving" in s for s in lanes_str):
        return "on_off_on_off"
    return "on_off"


# Per-geometry descriptive labels for each logical lane index.
# These MUST match the lane-index ordering in _build_freeway_lane_lists().
#   on_off            : 0 = single on/off ramp,          1..6 = mainline
#   on_off_on_off     : 0 = ramp 1 (on + off),           1 = ramp 2 (on + off),
#                       2..6 = mainline
#   on_on_off_off     : 0 = On-Ramp 1 taper only,
#                       1 = ramp weaving lane (On-Ramp 2 -> Off-Ramp 1),
#                       2 = Off-Ramp 2 taper only,
#                       3..8 = mainline
_FREEWAY_LANE_LABELS = {
    "on_off": {
        0: "Lane 0 (On/Off-Ramp)",
    },
    "on_off_on_off": {
        0: "Lane 0 (Ramp 1 — On+Off)",
        1: "Lane 1 (Ramp 2 — On+Off)",
    },
    "on_on_off_off": {
        0: "Lane 0 (On-Ramp 1 / Taper)",
        1: "Lane 1 (Ramp Weaving: On-2 -> Off-1)",
        2: "Lane 2 (Off-Ramp 2 / Taper)",
    },
}


def _freeway_lane_label(freeway_type, lane_idx):
    """Return a descriptive label for a logical lane given the freeway geometry."""
    explicit = _FREEWAY_LANE_LABELS.get(freeway_type, {}).get(lane_idx)
    if explicit is not None:
        return explicit
    return f"Lane {lane_idx} (Mainline)"


def assign_logical_lane(df):
    """Map each row's SUMO lane (edge/lane id) to logical lane index using same lists as freeway_sim."""
    freeway_type = _infer_freeway_type(df)
    lane_lists = _build_freeway_lane_lists()
    all_lane_names = lane_lists.get(freeway_type, lane_lists["on_off"])
    seg_to_idx = _lane_segment_to_idx_from_lists(all_lane_names)

    def _get_lane_id(raw_lane):
        s = str(raw_lane).strip()
        if s in seg_to_idx:
            return seg_to_idx[s]
        # SUMO getLaneID can return "edge_laneIndex" (e.g. Input_0_0); try edge part
        if "_" in s:
            edge_part = s.rsplit("_", 1)[0]
            if edge_part in seg_to_idx:
                return seg_to_idx[edge_part]
        return -1

    df = df.copy()
    df["logical_lane"] = df["lane"].apply(_get_lane_id)
    df = df[df["logical_lane"] >= 0].copy()
    df["_freeway_type"] = freeway_type  # carry geometry tag forward
    df["lane_label"] = df["logical_lane"].apply(lambda x: _freeway_lane_label(freeway_type, x))
    return df


def is_arterial(df):
    """True if data looks like arterial (EB/WB main road) not freeway."""
    if df.empty or "lane" not in df.columns:
        return False
    lanes_str = df["lane"].astype(str)
    has_eb_wb = lanes_str.str.contains("EB", na=False).any() or lanes_str.str.contains("WB", na=False).any()
    has_freeway = lanes_str.str.contains("Input_", na=False).any() or lanes_str.str.contains("On_Ramp_0", na=False).any()
    return bool(has_eb_wb and not has_freeway)


def assign_logical_lane_arterial_main(df):
    """Keep only MAIN ROAD (EB, WB), assign logical_lane and lane_label. Matches multi_inter_sim arterial."""
    lane_str = df["lane"].astype(str)
    df = df.copy()
    df["direction"] = np.where(lane_str.str.contains("EB"), "EB", np.where(lane_str.str.contains("WB"), "WB", None))
    df = df.dropna(subset=["direction"]).copy()
    if df.empty:
        return df
    lane_str = df["lane"].astype(str)
    df["lane_num"] = lane_str.str.split("_").str[-1].astype(int)
    dir_offset = np.where(df["direction"] == "EB", 10, 20)
    df["logical_lane"] = dir_offset + df["lane_num"]
    df["lane_label"] = df["direction"] + " Lane " + df["lane_num"].astype(str)
    df = df.sort_values(["id", "time"]).reset_index(drop=True)
    df["block_id"] = ((df["logical_lane"] != df["logical_lane"].shift(1)) | (df["id"] != df["id"].shift(1))).cumsum()
    block_stats = df.groupby("block_id")["time"].agg(["min", "max"])
    block_stats["duration"] = block_stats["max"] - block_stats["min"]
    noise_blocks = block_stats[block_stats["duration"] < MIN_LANE_STAY].index
    first_blocks = df.drop_duplicates(subset=["id"], keep="first")["block_id"].values
    is_noise = df["block_id"].isin(noise_blocks) & ~df["block_id"].isin(first_blocks)
    if is_noise.any():
        df.loc[is_noise, "logical_lane"] = np.nan
        df["logical_lane"] = df.groupby("id")["logical_lane"].ffill().bfill()
    df["logical_lane"] = df["logical_lane"].fillna(0).astype(int)
    return df


def robust_lane_filter(df, min_duration=MIN_LANE_STAY):
    df = df.sort_values(["id", "time"]).reset_index(drop=True)
    
    df['block_id'] = (
        (df['logical_lane'] != df['logical_lane'].shift(1)) | 
        (df['id'] != df['id'].shift(1))
    ).cumsum()
    
    block_stats = df.groupby('block_id')['time'].agg(['min', 'max'])
    block_stats['duration'] = block_stats['max'] - block_stats['min']
    
    noise_block_ids = block_stats[block_stats['duration'] < min_duration].index
    
    is_noise = df['block_id'].isin(noise_block_ids)
    
    first_blocks = df.drop_duplicates(subset=['id'], keep='first')['block_id']
    is_noise = is_noise & (~df['block_id'].isin(first_blocks))

    if is_noise.sum() > 0:
        df.loc[is_noise, 'logical_lane'] = np.nan
        df['logical_lane'] = df.groupby('id')['logical_lane'].ffill()
        df['logical_lane'] = df.groupby('id')['logical_lane'].bfill()
    
    df['logical_lane'] = df['logical_lane'].fillna(0).astype(int)

    # Preserve the geometry tag that assign_logical_lane attached so we can
    # keep using geometry-aware lane labels after the filter pass.
    if "_freeway_type" in df.columns and not df["_freeway_type"].isna().all():
        fwy = df["_freeway_type"].dropna().iloc[0]
    else:
        fwy = _infer_freeway_type(df)
    df["lane_label"] = df["logical_lane"].apply(lambda x: _freeway_lane_label(fwy, x))
    return df

# ---------------------------
# 4) PLOTTING FUNCTIONS
# ---------------------------

# 508-compliant: line styles by vehicle type (all black)
TYPE_LINESTYLES = {
    "0": "-",       # SV: solid
    "1": "--",      # AV: dashed
    "2": "-.",      # HV: dashdot
    "3": ":",       # CAV: dotted
}

# Max time gap (seconds) within same lane to still connect points; larger = new segment (lane change / re-entry)
MAX_TIME_GAP_CONNECT = 1.0


def _contiguous_segments(times):
    """Split indices into contiguous segments where time diff <= MAX_TIME_GAP_CONNECT."""
    if len(times) < 2:
        return [(0, len(times))] if len(times) else []
    t = np.asarray(times)
    d = np.diff(t)
    breaks = np.where(d > MAX_TIME_GAP_CONNECT)[0] + 1
    starts = np.concatenate([[0], breaks])
    ends = np.concatenate([breaks, [len(t)]])
    return list(zip(starts, ends))


def plot_lane_subplot(ax, lane_data, title, marker_size=10, *, uniform_linestyle=None):
    """
    508-compliant trajectory plot: lines only (no scatter), all black.
    By default, line style distinguishes vehicle type (solid, dashed, dashdot, dotted).
    If uniform_linestyle is set (CS4 shockwave), all trajectories use that style.
    Lines are drawn per (id, contiguous stay in lane) so lane changes do not connect.
    """
    if len(lane_data) > 10000:
        lane_data = lane_data.iloc[::2].copy()
    else:
        lane_data = lane_data.copy()

    ids = lane_data["id"].unique()
    if len(ids) > 400:
        keep_ids = np.random.choice(ids, 400, replace=False)
        lane_data = lane_data[lane_data["id"].isin(keep_ids)]

    lane_data = lane_data.sort_values(["id", "time"]).reset_index(drop=True)
    color_508 = (0, 0, 0)  # black for all

    def _draw_group(group, linestyle):
        t = group["time"].values
        s = group["s"].values
        if len(t) < 2:
            return
        for (i, j) in _contiguous_segments(t):
            if j - i >= 2:
                ax.plot(
                    t[i:j], s[i:j],
                    color=color_508,
                    linestyle=linestyle,
                    linewidth=0.8,
                    alpha=0.9,
                    solid_capstyle="round",
                )

    if uniform_linestyle is not None:
        for _, group in lane_data.groupby("id"):
            _draw_group(group, uniform_linestyle)
    else:
        for v_type in lane_data["type"].unique():
            v_type_str = str(v_type)
            subset = lane_data[lane_data["type"] == v_type]
            linestyle = TYPE_LINESTYLES.get(v_type_str, "-")
            for _, group in subset.groupby("id"):
                _draw_group(group, linestyle)

    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_ylabel("Pos (m)")
    ax.grid(True, alpha=0.3)

def _is_cs4_trajectory_label(label):
    if label in CS4_LABELS:
        return True
    meta = FILE_META.get(label, {})
    return meta.get("case") == "CS4"


def plot_logical_lane_trajectories(df, label, out_dir=OUT_DIR, *, run_slug=None):
    """If run_slug is set (e.g. CSV stem), saves {label}_{run_slug}_Trajectories.png per run."""
    cs4_plot = _is_cs4_trajectory_label(label)
    traj_linestyle = ":" if cs4_plot else None  # dotted for CS4 shockwave (SW_SV / SW_AV / SW_CAV)

    if is_arterial(df):
        df_raw = assign_logical_lane_arterial_main(df.copy())
        df_smooth = df_raw
    else:
        df_raw = assign_logical_lane(df.copy())
        df_smooth = robust_lane_filter(df_raw.copy()) 
    
    unique_lanes = sorted(df_smooth["logical_lane"].unique())
    n_lanes = len(unique_lanes)
    
    if n_lanes == 0:
        return ""

    nrows_per_lane = 2 if COMPARE_RAW_SMOOTHED else 1
    total_rows = n_lanes * nrows_per_lane
    row_height = TRAJ_ROW_HEIGHT_CS4_IN if cs4_plot else TRAJ_ROW_HEIGHT_IN

    fig, axes = plt.subplots(
        nrows=total_rows, ncols=1,
        figsize=(TRAJ_FIG_WIDTH_IN, row_height * total_rows),
        sharex=True,
    )
    if total_rows == 1: axes = [axes]

    # Geometry tag used as fallback when a lane subset has no lane_label yet.
    if "_freeway_type" in df_smooth.columns and not df_smooth["_freeway_type"].isna().all():
        _fwy_tag = df_smooth["_freeway_type"].dropna().iloc[0]
    else:
        _fwy_tag = _infer_freeway_type(df_smooth)

    for i, lane_idx in enumerate(unique_lanes):
        sub = df_smooth[df_smooth["logical_lane"] == lane_idx]
        if "lane_label" in sub.columns and not sub.empty:
            lane_label = sub["lane_label"].iloc[0]
        else:
            lane_label = _freeway_lane_label(_fwy_tag, lane_idx)
        base_ax_idx = i * nrows_per_lane
        
        if COMPARE_RAW_SMOOTHED:
            ax_raw = axes[base_ax_idx]
            raw_subset = df_raw[df_raw["logical_lane"] == lane_idx]
            plot_lane_subplot(
                ax_raw, raw_subset, f"{lane_label} [RAW DATA - NOISY]",
                marker_size=1, uniform_linestyle=traj_linestyle,
            )
            ax_raw.set_facecolor("#FFF0F0") 

            ax_smooth = axes[base_ax_idx + 1]
            smooth_subset = df_smooth[df_smooth["logical_lane"] == lane_idx]
            plot_lane_subplot(
                ax_smooth, smooth_subset, f"{lane_label} [SMOOTHED]",
                marker_size=1, uniform_linestyle=traj_linestyle,
            )
            ax_smooth.set_facecolor("#F0FFF0")
        else:
            ax = axes[i]
            smooth_subset = df_smooth[df_smooth["logical_lane"] == lane_idx]
            plot_lane_subplot(
                ax, smooth_subset, lane_label,
                marker_size=1, uniform_linestyle=traj_linestyle,
            )

    axes[-1].set_xlabel("Time (s)")
    
    if cs4_plot:
        legend_elements = [
            Line2D([0], [0], color="k", linestyle=":", linewidth=2, label="All vehicles (dotted)"),
        ]
        fig.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, 1.01), ncol=1, frameon=False)
    else:
        legend_elements = [
            Line2D([0], [0], color="k", linestyle=TYPE_LINESTYLES["0"], linewidth=2, label=TYPE_LABELS["0"]),
            Line2D([0], [0], color="k", linestyle=TYPE_LINESTYLES["1"], linewidth=2, label=TYPE_LABELS["1"]),
            Line2D([0], [0], color="k", linestyle=TYPE_LINESTYLES["2"], linewidth=2, label=TYPE_LABELS["2"]),
            Line2D([0], [0], color="k", linestyle=TYPE_LINESTYLES["3"], linewidth=2, label=TYPE_LABELS["3"]),
        ]
        fig.legend(handles=legend_elements, loc="upper center", bbox_to_anchor=(0.5, 1.01), ncol=4, frameon=False)

    plt.tight_layout()

    kind = "_Comparison.png" if COMPARE_RAW_SMOOTHED else "_Trajectories.png"
    if run_slug:
        filename = f"{label}_{run_slug}{kind}"
    else:
        filename = f"{label}{kind}"
    out_path = os.path.join(out_dir, filename)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved: {out_path}")
    return out_path

# ---------------------------
# 5) MAIN EXECUTION
# ---------------------------

def _avg_speed_and_tt(df):
    """Average speed (m/s) and per-vehicle travel time (s).

    For freeway, restrict to the homogeneous stretch [0, 2000] m so ramp entry/
    exit segments don't skew travel time. For arterial / intersection networks
    the spatial bounds differ, so we use the full trajectory span.
    """
    avg_spd = float(df["v"].mean()) if "v" in df.columns and not df.empty else float("nan")
    if is_arterial(df):
        d_tt = df
    else:
        d_tt = df[(df["s"] >= 0) & (df["s"] <= 2000)]
    if d_tt.empty:
        return avg_spd, float("nan")
    g = d_tt.groupby("id")["time"]
    tt_series = (g.max() - g.min())
    tt_valid = tt_series[tt_series > 0]
    avg_tt = float(tt_valid.mean()) if len(tt_valid) else float("nan")
    return avg_spd, avg_tt


def main():
    summary_data = []

    for label in iter_scenario_labels():
        meta = FILE_META.get(label, {"case": "", "env": "", "scenario": label})
        paths = discover_run_csv_paths(label)
        if not paths:
            folder = os.path.join(CASE_RUNS_DIR, label)
            leg = FILES_LEGACY.get(label, "")
            print(
                f"[SKIP] {label} ({meta.get('env','')}): no run_*.csv under {folder} "
                f"and no legacy file -> {leg}"
            )
            continue

        print(f"Processing {label} ({len(paths)} run(s)) ...")
        spd_runs, tt_runs = [], []
        for p in paths:
            df_run = load_trajectory_csv(p)
            s, t = _avg_speed_and_tt(df_run)
            spd_runs.append(s)
            tt_runs.append(t)

        avg_spd = float(np.nanmean(spd_runs))
        avg_tt = float(np.nanmean(tt_runs))
        std_spd = float(np.nanstd(spd_runs)) if len(spd_runs) > 1 else 0.0
        std_tt = float(np.nanstd(tt_runs)) if len(tt_runs) > 1 else 0.0

        # One trajectory figure per run (same order as discover_run_csv_paths); metrics above are averaged.
        traj_paths = []
        for p in paths:
            df_plot = load_trajectory_csv(p)
            if df_plot.empty:
                print(f"  [skip traj] empty or invalid: {p}")
                continue
            slug = os.path.splitext(os.path.basename(p))[0]
            outp = plot_logical_lane_trajectories(df_plot, label, run_slug=slug)
            if outp:
                traj_paths.append(outp)
        traj_note = "; ".join(traj_paths)

        summary_data.append({
            "Case":               meta["case"],
            "Environment":        meta["env"],
            "Scenario":           meta["scenario"],
            "Label":              label,
            "N_Runs":             len(paths),
            "Avg_Speed_m_s":      round(avg_spd, 2) if not np.isnan(avg_spd) else np.nan,
            "Avg_TravelTime_s":   round(avg_tt, 2) if not np.isnan(avg_tt) else np.nan,
            "Std_Speed_m_s":      round(std_spd, 4) if len(paths) > 1 and not np.isnan(std_spd) else np.nan,
            "Std_TravelTime_s":   round(std_tt, 4) if len(paths) > 1 and not np.isnan(std_tt) else np.nan,
            "Trajectory_Plot":    traj_note,
        })

    if not summary_data:
        print("No scenarios processed.")
        return

    sum_df = pd.DataFrame(summary_data)
    sum_path = os.path.join(OUT_DIR, "analysis_summary.csv")
    sum_df.to_csv(sum_path, index=False)
    print(f"\nAnalysis complete. Summary saved to: {sum_path}\n")

    # ------------- Grouped per-case-study printout -------------
    for case_id in active_case_study_ids():
        case_rows = sum_df[sum_df["Case"] == case_id]
        if case_rows.empty:
            continue
        print(f"===== {case_id} =====")
        cols = [
            "Environment",
            "Scenario",
            "N_Runs",
            "Avg_Speed_m_s",
            "Avg_TravelTime_s",
            "Std_Speed_m_s",
            "Std_TravelTime_s",
        ]
        print(case_rows[[c for c in cols if c in case_rows.columns]].to_string(index=False))
        print()


if __name__ == "__main__":
    main()

Processing CS1_HDV_Baseline (2 run(s)) ...
Saved: processed_outputs\CS1_HDV_Baseline_run_01_Trajectories.png
Saved: processed_outputs\CS1_HDV_Baseline_run_02_Trajectories.png
Processing CS1_Ideal (10 run(s)) ...
Saved: processed_outputs\CS1_Ideal_run_01_Trajectories.png
Saved: processed_outputs\CS1_Ideal_run_02_Trajectories.png
Saved: processed_outputs\CS1_Ideal_run_03_Trajectories.png
Saved: processed_outputs\CS1_Ideal_run_04_Trajectories.png
Saved: processed_outputs\CS1_Ideal_run_05_Trajectories.png
Saved: processed_outputs\CS1_Ideal_run_06_Trajectories.png
Saved: processed_outputs\CS1_Ideal_run_07_Trajectories.png
Saved: processed_outputs\CS1_Ideal_run_08_Trajectories.png
Saved: processed_outputs\CS1_Ideal_run_09_Trajectories.png
Saved: processed_outputs\CS1_Ideal_run_10_Trajectories.png
Processing CS1_HighLoss (10 run(s)) ...
Saved: processed_outputs\CS1_HighLoss_run_01_Trajectories.png
Saved: processed_outputs\CS1_HighLoss_run_02_Trajectories.png
Saved: processed_outputs\CS1_HighL

# Flow - Density Plots

In [19]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import sys

_NGM_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if _NGM_ROOT not in sys.path:
    sys.path.insert(0, _NGM_ROOT)
from ngm_paths import simulation_results_dir

# ============================
# CONFIGURATION
# ============================

DX = 200.0      # spatial bin width (m)
DT = 10.0        # time bin width (s)

S_MIN = 100.0   # homogeneous segment start (m)
S_MAX = 1500.0  # homogeneous segment end (m)

DENSITY_MAX = 150.0   # veh/km  (per-lane plot axis)
FLOW_MAX    = 2500.0  # veh/h   (per-lane plot axis)

CASE_RUNS_DIR = os.path.join(simulation_results_dir(), "case_runs")

# Case-study labels in registry order (matches FILE_META / run_case_studies.py)
CS1_FD_ORDER = [
    "CS1_HDV_Baseline", "CS1_Ideal", "CS1_HighLoss", "CS1_MidLatency_LowLoss",
    "CS1_MidLatency_MidLoss", "CS1_HighLatency", "CS1_HighLatency_HighLoss",
]
CS2_FWY_FD_ORDER = [
    "CS2_Fwy_100SV", "CS2_Fwy_90SV_10HV", "CS2_Fwy_80SV_20HV",
    "CS2_Fwy_45CAV_5CAHV_45SV_5HV", "CS2_Fwy_100CAV", "CS2_Fwy_90CAV_10CAHV", "CS2_Fwy_80CAV_20CAHV",
    "CS2_Fwy_90SV_10CAHV", "CS2_Fwy_90CAV_10HV",
]
CS2_ART_FD_ORDER = [
    "CS2_Art_90SV_10HV", "CS2_Art_80SV_20HV", "CS2_Art_90CAV_10CAHV", "CS2_Art_80CAV_20CAHV",
]

# Display label → subfolder under CASE_RUNS_DIR (respects CASE_STUDIES_TO_RUN from cell 1)
_fd_order = []
if "CS1" in ACTIVE_CASE_STUDIES:
    _fd_order.extend(CS1_FD_ORDER)
if "CS2" in ACTIVE_CASE_STUDIES:
    _fd_order.extend(CS2_FWY_FD_ORDER + CS2_ART_FD_ORDER)
CASES = {lbl: lbl for lbl in _fd_order}

# Single trajectory CSVs in CASE_RUNS_DIR (CS4 only)
_CASE_SINGLE_CSV_ALL = {
    "SW_SV": os.path.join(CASE_RUNS_DIR, "SW_SV.csv"),
    "SW_AV": os.path.join(CASE_RUNS_DIR, "SW_AV.csv"),
    "SW_CAV": os.path.join(CASE_RUNS_DIR, "SW_CAV.csv"),
}
CASE_SINGLE_CSV = {
    lbl: path
    for lbl, path in _CASE_SINGLE_CSV_ALL.items()
    if lbl in FILE_META and FILE_META[lbl].get("case") in ACTIVE_CASE_STUDIES
}

OUT_DIR = "processed_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================
# DATA LOADING
# ============================

def _load_single_csv(path):
    df = pd.read_csv(path)
    required = ["time", "id", "lane", "v"]
    if not all(c in df.columns for c in required):
        return pd.DataFrame()
    pos_col = "x" if "x" in df.columns else "lane_pos"
    if pos_col not in df.columns:
        return pd.DataFrame()
    df = df.dropna(subset=required + [pos_col])
    df = df.sort_values(["id", "time"]).reset_index(drop=True)
    df["time"] = pd.to_numeric(df["time"])
    df["id"]   = pd.to_numeric(df["id"])
    df["v"]    = pd.to_numeric(df["v"])
    df["s"]    = pd.to_numeric(df[pos_col])
    return df


def load_case_runs(case_dir):
    """Load all run_*.csv files from a case folder, tagging each with a run_id."""
    run_files = sorted(glob.glob(os.path.join(case_dir, "run_*.csv")))
    if not run_files:
        print(f"  [WARN] No run_*.csv files found in {case_dir}")
        return pd.DataFrame()
    dfs = []
    for run_id, path in enumerate(run_files):
        df = _load_single_csv(path)
        if df.empty:
            continue
        df["run_id"] = run_id
        dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    out = pd.concat(dfs, ignore_index=True)
    print(f"  Loaded {len(run_files)} runs, {len(out):,} rows total.")
    return out


# ============================
# LANE ASSIGNMENT
# ============================

def is_arterial(df):
    if df.empty or "lane" not in df.columns:
        return False
    lanes_str = df["lane"].astype(str)
    has_eb_wb  = lanes_str.str.contains("EB", na=False).any() or lanes_str.str.contains("WB", na=False).any()
    has_fwy    = lanes_str.str.contains("Input_", na=False).any() or lanes_str.str.contains("On_Ramp_0", na=False).any()
    return bool(has_eb_wb and not has_fwy)


def assign_logical_lane_freeway(df):
    """Mainline only: use proper lane lists, keep only mainline logical lanes."""
    freeway_type = _infer_freeway_type(df)
    lane_lists = _build_freeway_lane_lists()
    all_lane_names = lane_lists.get(freeway_type, lane_lists["on_off"])
    seg_to_idx = _lane_segment_to_idx_from_lists(all_lane_names)
    mainline_start = {"on_off": 1, "on_off_on_off": 2, "on_on_off_off": 3}.get(freeway_type, 1)

    def _get_lane_id(raw_lane):
        s = str(raw_lane).strip()
        if s in seg_to_idx:
            return seg_to_idx[s]
        if "_" in s:
            edge_part = s.rsplit("_", 1)[0]
            if edge_part in seg_to_idx:
                return seg_to_idx[edge_part]
        return -1

    df = df.copy()
    df["logical_lane"] = df["lane"].apply(_get_lane_id)
    df["_freeway_type"] = freeway_type
    df["lane_label"] = df["logical_lane"].apply(lambda x: _freeway_lane_label(freeway_type, x))
    return df[df["logical_lane"] >= mainline_start].copy()


def assign_logical_lane_arterial(df):
    """EB/WB main road only."""
    lane_str = df["lane"].astype(str)
    df = df.copy()
    df["direction"] = np.where(lane_str.str.contains("EB"), "EB",
                      np.where(lane_str.str.contains("WB"), "WB", None))
    df = df.dropna(subset=["direction"]).copy()
    if df.empty:
        return df
    df["lane_num"]     = df["lane"].astype(str).str.split("_").str[-1].astype(int)
    dir_offset         = np.where(df["direction"] == "EB", 10, 20)
    df["logical_lane"] = dir_offset + df["lane_num"]
    return df


# ============================
# EDIE FD — PER-RUN THEN COMBINE
# ============================

def _edie_bins_for_run(run_df, dx, dt):
    """Compute Edie space-time bins for one run (time reset to 0 within the run)."""
    run_df = run_df[(run_df["s"] >= S_MIN) & (run_df["s"] <= S_MAX)].copy()
    if run_df.empty:
        return pd.DataFrame()

    t0 = run_df["time"].min()
    run_df["t_rel"] = run_df["time"] - t0

    # Time to next observation of the same vehicle
    run_df["dt_raw"] = run_df.groupby("id")["t_rel"].shift(-1) - run_df["t_rel"]
    run_df["dt_raw"] = run_df["dt_raw"].clip(lower=0.0, upper=dt)

    run_df["tbin"] = np.floor(run_df["t_rel"] / dt).astype(int)
    run_df["sbin"] = np.floor((run_df["s"] - S_MIN) / dx).astype(int)

    # Clip dt so it doesn't cross the time-bin boundary
    t_bin_end       = (run_df["tbin"] + 1) * dt
    run_df["dt"]    = np.minimum(run_df["dt_raw"], t_bin_end - run_df["t_rel"]).clip(lower=0.0)
    run_df["ds"]    = run_df["v"] * run_df["dt"]

    agg = (
        run_df.groupby(["logical_lane", "tbin", "sbin"])
        .agg(time_spent=("dt", "sum"), dist_traveled=("ds", "sum"))
        .reset_index()
    )
    return agg


def edie_fd(df, label, dx=DX, dt=DT, density_lim=DENSITY_MAX, flow_lim=FLOW_MAX):
    """
    Edie (1965) fundamental diagram from trajectory data.
    Runs are processed independently (time reset per run) then combined for richer scatter.
    """
    df = df[(df["s"] >= S_MIN) & (df["s"] <= S_MAX)].copy()
    if df.empty:
        print(f"  [{label}] No data in range {S_MIN}–{S_MAX} m.")
        return

    fwy_type = df["_freeway_type"].iloc[0] if "_freeway_type" in df.columns else "on_off"
    n_lanes = df["logical_lane"].nunique()
    n_runs  = df["run_id"].nunique()
    cell_area = dx * dt

    # Edie bins per run → concatenate
    all_agg = []
    for run_id, run_df in df.groupby("run_id"):
        agg = _edie_bins_for_run(run_df, dx, dt)
        if agg.empty:
            continue
        agg["run_id"] = run_id
        all_agg.append(agg)

    if not all_agg:
        print(f"  [{label}] No bins computed.")
        return

    agg_all = pd.concat(all_agg, ignore_index=True)
    agg_all["density_veh_km"] = agg_all["time_spent"]    / cell_area * 1000.0
    agg_all["flow_veh_h"]     = agg_all["dist_traveled"] / cell_area * 3600.0

    # ---- Per-lane FD plots ----
    for lane in sorted(agg_all["logical_lane"].unique()):
        d = agg_all[agg_all["logical_lane"] == lane]
        fig, ax = plt.subplots(figsize=(7, 5))
        ax.scatter(d["density_veh_km"], d["flow_veh_h"],
                   s=12, alpha=0.4, edgecolors="none")
        ax.set_title(
            f"Fundamental Diagram – {_freeway_lane_label(fwy_type, lane)}\n"
            f"Edie (dx={dx} m, dt={dt} s) | {label} ({n_runs} runs)",
            fontsize=11, fontweight="bold"
        )
        ax.set_xlabel("Density (veh/km)")
        ax.set_ylabel("Flow (veh/h)")
        ax.set_xlim(0, density_lim)
        ax.set_ylim(0, flow_lim)
        ax.grid(True, alpha=0.3)
        out = os.path.join(OUT_DIR, f"{label}_Lane{lane}_FD.png")
        fig.savefig(out, dpi=150, bbox_inches="tight")
        plt.close(fig)

    # ---- Whole-highway FD (sum lanes per cell per run) ----
    agg_hw = (
        agg_all.groupby(["run_id", "tbin", "sbin"])
        .agg(time_spent=("time_spent", "sum"), dist_traveled=("dist_traveled", "sum"))
        .reset_index()
    )
    agg_hw["density_veh_km"] = agg_hw["time_spent"]    / cell_area * 1000.0
    agg_hw["flow_veh_h"]     = agg_hw["dist_traveled"] / cell_area * 3600.0

    fig, ax = plt.subplots(figsize=(7, 5))
    ax.scatter(agg_hw["density_veh_km"], agg_hw["flow_veh_h"],
               s=12, alpha=0.4, color="purple", edgecolors="none")
    ax.set_title(
        f"Fundamental Diagram – Mainline Total ({n_lanes} Lanes)\n"
        f"Edie (dx={dx} m, dt={dt} s) | {label} ({n_runs} runs)",
        fontsize=11, fontweight="bold"
    )
    ax.set_xlabel("Density (veh/km)")
    ax.set_ylabel("Flow (veh/h)")
    ax.set_xlim(0, 250)
    ax.set_ylim(0, 6000)
    ax.grid(True, alpha=0.3)
    out_hw = os.path.join(OUT_DIR, f"{label}_Mainline_FD.png")
    fig.savefig(out_hw, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  Saved FD: {out_hw}")


# ============================
# MAIN
# ============================

def _fd_pipeline(label, df):
    """Assign logical lanes and build FD plots for one loaded dataframe."""
    if df.empty:
        print("  -> No data loaded.")
        return
    if is_arterial(df):
        df = assign_logical_lane_arterial(df)
    else:
        df = assign_logical_lane_freeway(df)
    if df.empty:
        print("  -> No mainline rows after lane assignment.")
        return
    edie_fd(df, label)


def _fd_panel_subtitle(label: str) -> str:
    if "FILE_META" in globals() and label in FILE_META:
        return str(FILE_META[label].get("scenario", label))
    return label


def merge_mainline_fd_row(labels, *, title, out_file, ncol=None):
    """Stitch per-scenario *_Mainline_FD.png plots side by side for one case-study group."""
    from matplotlib import image as mpimg

    entries = []
    for lab in labels:
        path = os.path.join(OUT_DIR, f"{lab}_Mainline_FD.png")
        if not os.path.isfile(path):
            print(f"[panel] missing: {path}")
            continue
        entries.append((path, _fd_panel_subtitle(lab)))

    if not entries:
        print(f"[panel] {title}: no images found — skip")
        return

    n = len(entries)
    ncol = ncol or n
    nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 4.2 * nrow), squeeze=False)
    axes_flat = axes.flatten()

    for ax in axes_flat[n:]:
        ax.axis("off")

    for ax, (path, subtitle) in zip(axes_flat[:n], entries):
        ax.imshow(mpimg.imread(path))
        ax.set_title(subtitle, fontsize=8)
        ax.axis("off")

    fig.suptitle(title, fontsize=12, fontweight="bold")
    fig.tight_layout()
    out_path = os.path.join(OUT_DIR, out_file)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved FD panel: {out_path}")


def build_case_study_fd_panels():
    """After individual Mainline FD plots exist, merge by case study."""
    if "CS1" in ACTIVE_CASE_STUDIES:
        merge_mainline_fd_row(
            CS1_FD_ORDER,
            title="Case Study 1 — Mainline flow–density (V2V communication)",
            out_file="CS1_Mainline_FD_panel.png",
            ncol=6,
        )
    if "CS2" in ACTIVE_CASE_STUDIES:
        merge_mainline_fd_row(
            CS2_FWY_FD_ORDER,
            title="Case Study 2 — Freeway mainline flow–density (MPR)",
            out_file="CS2_Freeway_Mainline_FD_panel.png",
            ncol=9,
        )
        merge_mainline_fd_row(
            CS2_ART_FD_ORDER,
            title="Case Study 2 — Arterial mainline flow–density (MPR)",
            out_file="CS2_Arterial_Mainline_FD_panel.png",
            ncol=4,
        )


def main():
    for label, folder in CASES.items():
        case_dir = os.path.join(CASE_RUNS_DIR, folder)
        if not os.path.isdir(case_dir):
            print(f"[SKIP] Not found: {case_dir}")
            continue
        print(f"\nProcessing {label} ...")
        df = load_case_runs(case_dir)
        _fd_pipeline(label, df)

    for label, path in CASE_SINGLE_CSV.items():
        if not os.path.isfile(path):
            print(f"[SKIP] Not found: {path}")
            continue
        print(f"\nProcessing {label} (single CSV) ...")
        df = _load_single_csv(path)
        if df.empty:
            print("  -> No data loaded.")
            continue
        df = df.copy()
        df["run_id"] = 0
        _fd_pipeline(label, df)

    print("\n--- Merging Mainline FD panels by case study ---")
    build_case_study_fd_panels()


main()


Processing CS1_HDV_Baseline ...
  Loaded 2 runs, 440,703 rows total.
  Saved FD: processed_outputs\CS1_HDV_Baseline_Mainline_FD.png

Processing CS1_Ideal ...
  Loaded 10 runs, 2,352,713 rows total.
  Saved FD: processed_outputs\CS1_Ideal_Mainline_FD.png

Processing CS1_HighLoss ...
  Loaded 10 runs, 2,337,837 rows total.
  Saved FD: processed_outputs\CS1_HighLoss_Mainline_FD.png

Processing CS1_MidLatency_LowLoss ...
  Loaded 10 runs, 2,354,166 rows total.
  Saved FD: processed_outputs\CS1_MidLatency_LowLoss_Mainline_FD.png

Processing CS1_MidLatency_MidLoss ...
  Loaded 10 runs, 2,359,120 rows total.
  Saved FD: processed_outputs\CS1_MidLatency_MidLoss_Mainline_FD.png

Processing CS1_HighLatency ...
  Loaded 10 runs, 2,322,161 rows total.
  Saved FD: processed_outputs\CS1_HighLatency_Mainline_FD.png

Processing CS1_HighLatency_HighLoss ...
  Loaded 10 runs, 2,366,307 rows total.
  Saved FD: processed_outputs\CS1_HighLatency_HighLoss_Mainline_FD.png

--- Merging Mainline FD panels by 

# Case Study 3 — Safety (Time-to-Collision) Analysis

Single-intersection environment, 50/50 SV + CAV traffic.

Group A (Human Drivers) — type `SV`, governed by IDM + MOBIL.
Group B (CAVs) — type `CAV`, governed by Cooperative IDM + Cooperative MOBIL.

Safety is quantified by the Time-to-Collision (TTC) distribution, comparing the frequency of critical safety events (TTC below an interaction-specific threshold) between the two groups:

- **V2V (car–car):** critical TTC **1.5 s**
- **V2Ped (vehicle–pedestrian):** critical TTC **1.2 s**
- **V2Bike (vehicle–bicycle):** critical TTC **1.8 s**

We report **four** motor-related interaction types (plus V2P / V2B):

- **V2V-1D (link):** Same-lane rear-end TTC on **approach / link** lanes only (SUMO lanes **not** starting with ``:``).
- **V2V-2D (intersection):** Predictive **2-D** motor–motor TTC when **at least one** vehicle is on an **internal** junction lane (``lane`` id starts with ``:``).
- Requires string **`lane`** (never coerce to numeric) plus **`lane_pos`** for 1-D; **`x`, `y`, `theta`** for 2-D.
- Fleet arms per scenario: **SV vs CAV** or **SV vs AV** (`CS3_TTC_COMPARE`).
- **V2P / V2B (vehicle–pedestrian / vehicle–bicycle):** For each **motor-vehicle** vs **pedestrian (type -1)** or **bicycle (type -2)** **ID pair**, **one** TTC = **minimum** over the run using a 2-D **radial** time-to-collision (requires `x`, `y`, `theta` in the export; angle convention matches `single_inter_sim`).
- **Directness filter:** V2P/V2B TTC is computed only when the pair is within `TTC_VRU_MAX_RANGE_M` and there is **no other motor vehicle geometrically blocking** the line segment from vehicle to VRU (constants `TTC_DIRECT_BLOCK_PERP_M`, `TTC_DIRECT_BLOCK_ALONG_MIN_M`).

Distributions and the summary table use **scenario-specific arms** from `CS3_TTC_COMPARE` (SV vs CAV, or SV vs AV), using **follower** type for V2V and **subject vehicle** type for V2P/V2B.

### Part 1 vs Part 2

- **Part 1 (unchanged):** Pooled CS3 summary tables — all movements combined. Report CDFs: **four** combined plots (`CS3_SV_CAV_AV_*_TTC_CDF.png`) with **SV, CAV, AV** on one axis per interaction (SV/AV from `CS3_50SV_50AV`, CAV from `CS3_50SV_50CAV`).
- **Part 2 (new):** Movement × type tables only. Each motor vehicle’s **intended movement** comes from O–D in `models/all_trips.trips.xml` (`from` / `to`, same mapping as `single_inter_sim.py`: **Left Turn**, **Right Turn**, **Through**). Those movements are **not merged** in Part 2; each is reported separately with the same type arms as Part 1.  
  Output: `{label}_TTC_by_movement_type_summary.csv`.

In [21]:
# Pasted into Results_Processing.ipynb — Case Study 3 TTC cell (source is Python).
import os
import xml.etree.ElementTree as ET
from typing import Dict, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================================
# CONFIGURATION
# ==========================================================================
TTC_MAX_PLOT = 15.0
# Interaction-specific critical TTC thresholds (s)
TTC_CRITICAL_V2V = 1.5
TTC_CRITICAL_V2PED = 1.2
TTC_CRITICAL_V2BIKE = 1.8
TTC_CRITICAL_BY_INTERACTION = {
    "V2V-1D (link)": TTC_CRITICAL_V2V,
    "V2V-2D (intersection)": TTC_CRITICAL_V2V,
    "V2Ped": TTC_CRITICAL_V2PED,
    "V2Bike": TTC_CRITICAL_V2BIKE,
}
DEFAULT_LENGTH = 5.0


def _ttc_critical(interaction: str) -> float:
    return TTC_CRITICAL_BY_INTERACTION.get(interaction, TTC_CRITICAL_V2V)


# Per CS3 label: which agent types (single_inter_sim) go in each comparison arm.
CS3_TTC_COMPARE = {
    "CS3_50SV_50CAV": (
        ("Human", frozenset({"0", "2"})),   # SV, HV — IDM / DDM
        ("CAV", frozenset({"3", "4"})),     # CAV, CAHV — C-IDM / C-MOBIL
    ),
    "CS3_50SV_50AV": (
        ("SV", frozenset({"0"})),           # standard vehicle
        ("AV", frozenset({"1"})),           # automated (PT / DDM, not cooperative)
    ),
}

# Report figures: one CDF per interaction with SV, CAV, AV on the same axes.
CS3_SAVE_PER_SCENARIO_TTC_PLOTS = False
CS3_COMBINED_FLEET_SPEC = {
    "SV": ("CS3_50SV_50AV", frozenset({"0"})),
    "AV": ("CS3_50SV_50AV", frozenset({"1"})),
    "CAV": ("CS3_50SV_50CAV", frozenset({"3", "4"})),
}
CS3_COMBINED_FLEET_STYLE = {
    "SV": ("#555555", "-"),
    "AV": ("#0072B2", "--"),
    "CAV": ("#009E73", ":"),
}
CS3_COMBINED_PLOT_PREFIX = "CS3_SV_CAV_AV"

# Part 2 — O-D movement labels (match single_inter_sim.py)
CS3_INTER_IN_NAMES = ["EB_West", "WB_East", "SB_North", "NB_South"]
CS3_INTER_LT_NAMES = ["NB_North", "SB_South", "EB_East", "WB_West"]
CS3_INTER_RT_NAMES = ["SB_South", "NB_North", "WB_West", "EB_East"]
CS3_INTER_THROUGH_NAMES = ["EB_East", "WB_West", "SB_South", "NB_North"]
CS3_TTC_PART2_MOVEMENTS = ("Left Turn", "Right Turn", "Through")

MODELS_DIR = os.path.join(os.path.dirname(RESULTS_DIR), "models")
CS3_TRIPS_XML = os.path.join(MODELS_DIR, "all_trips.trips.xml")

import sys as _sys
if RESULTS_DIR not in _sys.path:
    _sys.path.insert(0, RESULTS_DIR)
from safety_metrics import (
    compute_ttc_v2v_1d,
    compute_ttc_v2v_2d_intersection,
    diagnose_ttc_v2v_1d,
)
# single_inter_sim: agent_type -1 = pedestrian, -2 = bicycle (in CSV "type" column)
PED_TYPE = -1
BIKE_TYPE = -2
# Skip vehicle–VRU pairs farther than this (m); TTC is only interpretable locally.
TTC_VRU_MAX_RANGE_M = 8.0
# Enforce "direct" V2VRU TTC: reject if another motor vehicle blocks line segment vehicle->VRU.
TTC_DIRECT_BLOCK_PERP_M = 2.5
TTC_DIRECT_BLOCK_ALONG_MIN_M = 1.5
OUT_DIR = "processed_outputs"
os.makedirs(OUT_DIR, exist_ok=True)


def _type_int(s):
    try:
        return int(float(str(s).split(".")[0]))
    except (ValueError, TypeError):
        return 999


def _is_motor(s) -> bool:
    return _type_int(s) >= 0


def _is_ped(s) -> bool:
    return _type_int(s) == PED_TYPE


def _is_bike(s) -> bool:
    return _type_int(s) == BIKE_TYPE


def sumo_angle_to_rad(sumo_deg) -> float:
    """Match single_inter_sim: SUMO angle in degrees to radians for cos/sin of velocity."""
    phi_deg = 90.0 - float(np.asarray(sumo_deg, dtype=float))
    phi_deg = np.fmod(phi_deg + 360.0, 360.0)
    return float(np.radians(phi_deg))


def vel_from_sumo(v, theta_sumo):
    th = sumo_angle_to_rad(theta_sumo)
    return float(v) * np.cos(th), float(v) * np.sin(th)


def ttc_2d_radial(
    x0, y0, vx0, vy0, x1, y1, vx1, vy1, max_ttc: float = 1.0e4
) -> float:
    """
    Radial (range) TTC from vehicle 0 to point 1: |r| / (-d|r|/dt) when d|r|/dt < 0.
    r = p1 - p0, d|r|/dt = u · (v1 - v0) with u = r/|r|.
    """
    rx = x1 - x0
    ry = y1 - y0
    dist = float(np.hypot(rx, ry))
    if dist < 1e-4:
        return float("nan")
    ux, uy = rx / dist, ry / dist
    vrx, vry = float(vx1 - vx0), float(vy1 - vy0)
    dr_dt = vrx * ux + vry * uy
    if dr_dt >= -1e-9:
        return float("nan")
    ttc = dist / (-dr_dt)
    if (not np.isfinite(ttc)) or ttc < 0.0 or ttc > max_ttc:
        return float("nan")
    return float(ttc)


def is_direct_vehicle_vru_pair(x0, y0, x1, y1, blockers_xy) -> bool:
    """True if no motor vehicle lies on the segment vehicle->VRU (simple occlusion gate)."""
    segx = x1 - x0
    segy = y1 - y0
    seg_len = float(np.hypot(segx, segy))
    if seg_len <= 1e-6:
        return False
    ux, uy = segx / seg_len, segy / seg_len

    for bx, by in blockers_xy:
        dx = bx - x0
        dy = by - y0
        along = dx * ux + dy * uy
        if along <= TTC_DIRECT_BLOCK_ALONG_MIN_M or along >= seg_len - TTC_DIRECT_BLOCK_ALONG_MIN_M:
            continue
        perp = abs(dx * uy - dy * ux)
        if perp <= TTC_DIRECT_BLOCK_PERP_M:
            return False
    return True


# ==========================================================================
# LOADING
# ==========================================================================
def load_csv_for_ttc(path: str) -> pd.DataFrame:
    """Load trajectories. V2V uses motor vehicles on non-internal lanes; V2P/V2B use x, y, theta."""
    df = pd.read_csv(path)
    for col in ("time", "id", "v"):
        if col not in df.columns:
            raise ValueError(f"TTC: missing required column {col!r} in {path}")
        df[col] = pd.to_numeric(df[col], errors="coerce")
    if "type" in df.columns:
        df["type"] = df["type"].astype(str).str.split(".").str[0]
    else:
        df["type"] = "0"
    if "lane" in df.columns:
        df["lane"] = df["lane"].astype(str)
    for col in ("lane_pos", "x", "y", "theta", "length"):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "length" not in df.columns:
        df["length"] = DEFAULT_LENGTH
    else:
        df["length"] = df["length"].fillna(DEFAULT_LENGTH)
    df = df.dropna(subset=["time", "id", "v"])
    return df




# ==========================================================================
# V2Ped / V2Bike: 2D radial TTC, min per (veh_id, vru_id)
# ==========================================================================
def compute_ttc_v2vru(df: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if not {"x", "y", "theta"}.issubset(df.columns):
        return (
            pd.DataFrame(columns=["veh_id", "vru_id", "vru_class", "ttc", "veh_type"]),
            pd.DataFrame(columns=["veh_id", "vru_id", "vru_class", "ttc", "veh_type"]),
        )
    w = df.dropna(subset=["x", "y", "theta", "time", "id"]).copy()
    if w.empty:
        return (
            pd.DataFrame(columns=["veh_id", "vru_id", "vru_class", "ttc", "veh_type"]),
            pd.DataFrame(columns=["veh_id", "vru_id", "vru_class", "ttc", "veh_type"]),
        )
    rows = []
    for _t, g in w.groupby("time"):
        V = g[g["type"].map(_is_motor)]
        P = g[g["type"].map(_is_ped)]
        B = g[g["type"].map(_is_bike)]
        if V.empty or (P.empty and B.empty):
            continue
        rmax = TTC_VRU_MAX_RANGE_M
        motor_xy = {int(r["id"]): (float(r["x"]), float(r["y"])) for _, r in V.iterrows()}
        for _, rv in V.iterrows():
            try:
                vx, vy = vel_from_sumo(rv["v"], rv["theta"])
            except (TypeError, ValueError, ZeroDivisionError):
                continue
            x0, y0 = float(rv["x"]), float(rv["y"])
            idv, tv = int(rv["id"]), str(rv["type"])
            blockers = [xy for vid, xy in motor_xy.items() if vid != idv]
            for S, vclass in ((P, "ped"), (B, "bike")):
                if S.empty:
                    continue
                for _, rp in S.iterrows():
                    x1, y1 = float(rp["x"]), float(rp["y"])
                    if (x1 - x0) ** 2 + (y1 - y0) ** 2 > rmax**2:
                        continue
                    if not is_direct_vehicle_vru_pair(x0, y0, x1, y1, blockers):
                        continue
                    try:
                        vpx, vpy = vel_from_sumo(rp["v"], rp["theta"])
                    except (TypeError, ValueError, ZeroDivisionError):
                        continue
                    ttc = ttc_2d_radial(x0, y0, vx, vy, x1, y1, vpx, vpy)
                    if np.isfinite(ttc):
                        rows.append((idv, int(rp["id"]), vclass, ttc, tv))
    cols = ["veh_id", "vru_id", "vru_class", "ttc", "veh_type"]
    if not rows:
        empty = pd.DataFrame(columns=cols)
        return empty, empty
    raw = pd.DataFrame(rows, columns=cols)

    def _agg_sub(cls: str) -> pd.DataFrame:
        sub = raw[raw["vru_class"] == cls]
        if sub.empty:
            return pd.DataFrame(columns=cols)
        return sub.groupby(["veh_id", "vru_id"], as_index=False).agg(
            ttc=("ttc", "min"),
            vru_class=("vru_class", "first"),
            veh_type=("veh_type", "first"),
        )

    return _agg_sub("ped"), _agg_sub("bike")


def _compare_arms(label: str):
    if label not in CS3_TTC_COMPARE:
        raise KeyError(f"No TTC compare arms for {label!r}; add to CS3_TTC_COMPARE.")
    return CS3_TTC_COMPARE[label]


def classify_cs3_vehicle_movement(origin: str, destination: str) -> str:
    """Map SUMO trip from/to edges to Left Turn / Right Turn / Through."""
    o, d = str(origin).strip(), str(destination).strip()
    if o not in CS3_INTER_IN_NAMES:
        return "Other"
    idx = CS3_INTER_IN_NAMES.index(o)
    if d == CS3_INTER_LT_NAMES[idx]:
        return "Left Turn"
    if d == CS3_INTER_RT_NAMES[idx]:
        return "Right Turn"
    if d == CS3_INTER_THROUGH_NAMES[idx]:
        return "Through"
    return "Other"


def load_cs3_vehicle_movement_map(trips_xml: str = CS3_TRIPS_XML) -> Dict[str, str]:
    """vehicle id (str) -> movement label from single_inter route file."""
    out: Dict[str, str] = {}
    if not os.path.isfile(trips_xml):
        print(f"[CS3 TTC Part 2] trips XML not found: {trips_xml}")
        return out
    try:
        root = ET.parse(trips_xml).getroot()
    except ET.ParseError as ex:
        print(f"[CS3 TTC Part 2] failed to parse trips XML: {ex}")
        return out
    for elem in root:
        if elem.tag != "trip":
            continue
        vid = elem.get("id")
        if vid is None:
            continue
        out[str(vid).split(".")[0]] = classify_cs3_vehicle_movement(
            elem.get("from", ""), elem.get("to", "")
        )
    return out


def _attach_movement(df: pd.DataFrame, id_col: str, mov_map: Dict[str, str]) -> pd.DataFrame:
    if df is None or df.empty:
        return df
    out = df.copy()
    out["movement"] = (
        out[id_col].astype(str).str.split(".").str[0].map(mov_map).fillna("Other")
    )
    return out


def _split_by_compare_movement(
    df: pd.DataFrame, type_col: str, movement: str, label: str,
) -> tuple[pd.Series, pd.Series, str, str]:
    (name_a, types_a), (name_b, types_b) = _compare_arms(label)
    if df is None or df.empty or "movement" not in df.columns:
        return pd.Series(dtype=float), pd.Series(dtype=float), name_a, name_b
    sub = df[df["movement"] == movement]
    if sub.empty:
        return pd.Series(dtype=float), pd.Series(dtype=float), name_a, name_b
    t = sub[type_col].astype(str).str.split(".").str[0]
    return sub.loc[t.isin(types_a), "ttc"], sub.loc[t.isin(types_b), "ttc"], name_a, name_b


def summarize_ttc_by_movement_and_type(
    ttc_v2v_1d: pd.DataFrame,
    ttc_v2v_2d: pd.DataFrame,
    ttc_ped: pd.DataFrame,
    ttc_bike: pd.DataFrame,
    label: str,
    mov_map: Dict[str, str],
    out_dir: str = OUT_DIR,
) -> pd.DataFrame:
    """Part 2: Interaction × Movement × Group; LT / RT / Through separate."""
    v2v_1d = _attach_movement(ttc_v2v_1d, "follower_id", mov_map)
    v2v_2d = _attach_movement(ttc_v2v_2d, "follower_id", mov_map)
    ped = _attach_movement(ttc_ped, "veh_id", mov_map)
    bike = _attach_movement(ttc_bike, "veh_id", mov_map)

    specs = (
        (v2v_1d, "follower_type", "V2V-1D (link)"),
        (v2v_2d, "follower_type", "V2V-2D (intersection)"),
        (ped, "veh_type", "V2Ped"),
        (bike, "veh_type", "V2Bike"),
    )
    all_rows: list[dict] = []
    for movement in CS3_TTC_PART2_MOVEMENTS:
        for df_i, tcol, interaction in specs:
            ha, hb, name_a, name_b = _split_by_compare_movement(df_i, tcol, movement, label)
            for nm, series in ((name_a, ha), (name_b, hb)):
                if series is None or series.empty:
                    thr = _ttc_critical(interaction)
                    all_rows.append({
                        "Interaction": interaction,
                        "Movement": movement,
                        "Group": nm,
                        "Pairs": 0,
                        "TTC_mean": np.nan,
                        "TTC_median": np.nan,
                        "TTC_p05": np.nan,
                        "Critical_threshold_s": thr,
                        "Critical": 0,
                        "Critical_rate_%": np.nan,
                    })
                    continue
                thr = _ttc_critical(interaction)
                crit = int((series < thr).sum())
                n = int(len(series))
                all_rows.append({
                    "Interaction": interaction,
                    "Movement": movement,
                    "Group": nm,
                    "Pairs": n,
                    "TTC_mean": round(float(series.mean()), 3),
                    "TTC_median": round(float(series.median()), 3),
                    "TTC_p05": round(float(series.quantile(0.05)), 3),
                    "Critical_threshold_s": thr,
                    "Critical": crit,
                    "Critical_rate_%": round(100.0 * crit / n, 3),
                })

    summary = pd.DataFrame(all_rows)
    print(f"\n[{label}] Part 2 — TTC by movement × type (LT / RT / Through separate):")
    print(summary.to_string(index=False))
    path = os.path.join(out_dir, f"{label}_TTC_by_movement_type_summary.csv")
    summary.to_csv(path, index=False)
    print(f"Saved movement×type TTC summary: {path}\n")
    return summary


def _split_by_compare(df: pd.DataFrame, type_col: str, label: str) -> tuple[pd.Series, pd.Series, str, str]:
    """Return (series_a, series_b, name_a, name_b) for this CS3 scenario."""
    (name_a, types_a), (name_b, types_b) = _compare_arms(label)
    if df is None or df.empty:
        return pd.Series(dtype=float), pd.Series(dtype=float), name_a, name_b
    t = df[type_col].astype(str).str.split(".").str[0]
    sa = df.loc[t.isin(types_a), "ttc"]
    sb = df.loc[t.isin(types_b), "ttc"]
    return sa, sb, name_a, name_b


def _summary_rows(series_a, series_b, interaction: str, name_a: str, name_b: str):
    rows = []
    thr = _ttc_critical(interaction)
    for name, series in [(name_a, series_a), (name_b, series_b)]:
        if series is None or series.empty or len(series) == 0:
            rows.append(
                {
                    "Interaction": interaction,
                    "Group": name,
                    "Pairs": 0,
                    "TTC_mean": np.nan,
                    "TTC_median": np.nan,
                    "TTC_p05": np.nan,
                    "Critical_threshold_s": thr,
                    "Critical": 0,
                    "Critical_rate_%": np.nan,
                }
            )
            continue
        crit = int((series < thr).sum())
        n = int(len(series))
        rows.append(
            {
                "Interaction": interaction,
                "Group": name,
                "Pairs": n,
                "TTC_mean": round(float(series.mean()), 3),
                "TTC_median": round(float(series.median()), 3),
                "TTC_p05": round(float(series.quantile(0.05)), 3),
                "Critical_threshold_s": thr,
                "Critical": crit,
                "Critical_rate_%": round(100.0 * crit / n, 3),
            }
        )
    return rows


def _series_for_fleet_types(
    df: pd.DataFrame, type_col: str, types: frozenset,
) -> pd.Series:
    if df is None or df.empty:
        return pd.Series(dtype=float)
    t = df[type_col].astype(str).str.split(".").str[0]
    return df.loc[t.isin(types), "ttc"]


def plot_cs3_combined_fleet_cdfs(
    pooled_by_label: dict[str, dict[str, pd.DataFrame]],
    out_dir: str = OUT_DIR,
) -> None:
    """Four report CDFs: SV, CAV, AV on one plot per interaction type."""
    specs = (
        ("v2v_1d", "follower_type", "V2V_1D_link", "V2V-1D (link)",
         "V2V-1D link (same-lane) — CS3"),
        ("v2v_2d", "follower_type", "V2V_2D_intersection", "V2V-2D (intersection)",
         "V2V-2D intersection — CS3"),
        ("ped", "veh_type", "V2Ped", "V2Ped",
         "V2P (vehicle–pedestrian) TTC — CS3"),
        ("bike", "veh_type", "V2Bike", "V2Bike",
         "V2B (vehicle–bicycle) TTC — CS3"),
    )
    any_data = False
    for key, tcol, stem, interaction, title in specs:
        series_by_fleet: list[tuple[str, pd.Series]] = []
        for fleet, (src_label, types) in CS3_COMBINED_FLEET_SPEC.items():
            bundle = pooled_by_label.get(src_label, {})
            df_i = bundle.get(key)
            s = _series_for_fleet_types(df_i, tcol, types)
            if s is not None and not s.empty:
                series_by_fleet.append((fleet, s))
        if not series_by_fleet:
            print(f"[CS3 combined] {stem}: no SV/CAV/AV data — skipped.")
            continue
        any_data = True
        thr = _ttc_critical(interaction)
        fig, ax = plt.subplots(figsize=(9, 5.5))
        for fleet, s in series_by_fleet:
            color, ls = CS3_COMBINED_FLEET_STYLE[fleet]
            srt = np.sort(s.values)
            cdf = np.arange(1, len(srt) + 1) / len(srt)
            ax.plot(
                srt, cdf, color=color, linestyle=ls, linewidth=2.0,
                label=f"{fleet} (n={len(s)})",
            )
        ax.axvline(
            thr, color="red", linestyle="--", linewidth=1.5,
            label=f"Critical TTC = {thr} s",
        )
        ax.set_xlabel("Time-to-Collision (s)")
        ax.set_ylabel("Empirical CDF")
        ax.set_title(f"{title} — CDF", fontsize=12, fontweight="bold")
        ax.set_xlim(0, TTC_MAX_PLOT)
        ax.set_ylim(0, 1.0)
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.tight_layout()
        cdf_path = os.path.join(out_dir, f"{CS3_COMBINED_PLOT_PREFIX}_{stem}_TTC_CDF.png")
        fig.savefig(cdf_path, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"Saved combined TTC CDF: {cdf_path}")
    if not any_data:
        print("[CS3 combined] No report CDFs written (missing pooled TTC data).")


def _hist_cdf_png(
    series_a, series_b, label: str, title: str, stem: str, out_dir: str,
    name_a: str, name_b: str, ttc_critical: float,
):
    bins = np.linspace(0, TTC_MAX_PLOT, 31)
    fig, ax = plt.subplots(figsize=(9, 5.5))
    if series_a is not None and (not series_a.empty):
        ax.hist(
            series_a.clip(upper=TTC_MAX_PLOT),
            bins=bins,
            alpha=0.55,
            color="#555555",
            edgecolor="black",
            label=f"{name_a} (n={len(series_a)})",
        )
    if series_b is not None and (not series_b.empty):
        ax.hist(
            series_b.clip(upper=TTC_MAX_PLOT),
            bins=bins,
            alpha=0.55,
            color="#009E73",
            edgecolor="black",
            label=f"{name_b} (n={len(series_b)})",
        )
    if (series_a is None or series_a.empty) and (series_b is None or series_b.empty):
        plt.close(fig)
        return
    ax.axvline(
        ttc_critical,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label=f"Critical TTC = {ttc_critical} s",
    )
    ax.set_xlabel("Time-to-Collision (s)")
    ax.set_ylabel("Count of unique pairs (min TTC per pair)")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlim(0, TTC_MAX_PLOT)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.tight_layout()
    hist_path = os.path.join(out_dir, f"{label}_{stem}_TTC_Distribution.png")
    fig.savefig(hist_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved TTC histogram: {hist_path}")

    fig2, ax2 = plt.subplots(figsize=(9, 5.5))
    for n2, c2, s2 in [
        (name_a, "#555555", series_a),
        (name_b, "#009E73", series_b),
    ]:
        if s2 is None or s2.empty:
            continue
        srt = np.sort(s2.values)
        cdf = np.arange(1, len(srt) + 1) / len(srt)
        ax2.plot(srt, cdf, color=c2, linewidth=2.0, label=n2)
    ax2.axvline(
        ttc_critical,
        color="red",
        linestyle="--",
        linewidth=1.5,
        label=f"Critical TTC = {ttc_critical} s",
    )
    ax2.set_xlabel("Time-to-Collision (s)")
    ax2.set_ylabel("Empirical CDF")
    ax2.set_title(f"{title} — CDF", fontsize=12, fontweight="bold")
    ax2.set_xlim(0, TTC_MAX_PLOT)
    ax2.set_ylim(0, 1.0)
    ax2.grid(True, alpha=0.3)
    ax2.legend()
    plt.tight_layout()
    cdf_path = os.path.join(out_dir, f"{label}_{stem}_TTC_CDF.png")
    fig2.savefig(cdf_path, dpi=150, bbox_inches="tight")
    plt.close(fig2)
    print(f"Saved TTC CDF:      {cdf_path}")


def plot_ttc_distributions(
    ttc_v2v_1d: pd.DataFrame,
    ttc_v2v_2d: pd.DataFrame,
    ttc_ped: pd.DataFrame,
    ttc_bike: pd.DataFrame,
    label: str,
    out_dir: str = OUT_DIR,
    save_plots: bool = CS3_SAVE_PER_SCENARIO_TTC_PLOTS,
):
    if save_plots:
        # --- V2V-1D link (same-lane, off intersection) ---
        ha, hb, name_a, name_b = _split_by_compare(ttc_v2v_1d, "follower_type", label)
        if ha.empty and hb.empty:
            print(f"[{label}] V2V-1D (link): no valid same-lane closing pairs.")
        else:
            _hist_cdf_png(
                ha, hb, label,
                f"V2V-1D link (same-lane) — {label}",
                "V2V_1D_link", out_dir, name_a, name_b,
                _ttc_critical("V2V-1D (link)"),
            )
        # --- V2V-2D intersection ---
        ha, hb, name_a, name_b = _split_by_compare(ttc_v2v_2d, "follower_type", label)
        if ha.empty and hb.empty:
            print(f"[{label}] V2V-2D (intersection): no valid predictive pairs on internal lanes.")
        else:
            _hist_cdf_png(
                ha, hb, label,
                f"V2V-2D intersection — {label}",
                "V2V_2D_intersection", out_dir, name_a, name_b,
                _ttc_critical("V2V-2D (intersection)"),
            )
        # --- V2P / V2B (subject vehicle type) ---
        for df_sub, stem, ttitle in (
            (ttc_ped, "V2Ped", "V2P (vehicle–pedestrian) TTC by vehicle type"),
            (ttc_bike, "V2Bike", "V2B (vehicle–bicycle) TTC by vehicle type"),
        ):
            if df_sub is None or df_sub.empty:
                print(f"[{label}] {stem}: no valid pairs (need x, y, theta in CSV, and VRU agents).")
                continue
            ha, hb, name_a, name_b = _split_by_compare(df_sub, "veh_type", label)
            if ha.empty and hb.empty:
                print(f"[{label}] {stem}: no {name_a}/{name_b} vehicle rows for this scenario mix.")
                continue
            _hist_cdf_png(
                ha, hb, label, f"{ttitle} — {label}", stem, out_dir, name_a, name_b,
                _ttc_critical(stem),
            )

    all_rows: list[dict] = []
    for df_i, tcol, interaction in (
        (ttc_v2v_1d, "follower_type", "V2V-1D (link)"),
        (ttc_v2v_2d, "follower_type", "V2V-2D (intersection)"),
        (ttc_ped, "veh_type", "V2Ped"),
        (ttc_bike, "veh_type", "V2Bike"),
    ):
        if df_i is None or df_i.empty:
            na, nb = _compare_arms(label)[0][0], _compare_arms(label)[1][0]
            all_rows.extend(_summary_rows(None, None, interaction, na, nb))
            continue
        ha, hb, name_a, name_b = _split_by_compare(df_i, tcol, label)
        all_rows.extend(_summary_rows(ha, hb, interaction, name_a, name_b))

    summary = pd.DataFrame(all_rows)
    print(f"\n[{label}] Safety summary (all interaction types):")
    print(summary.to_string(index=False))
    sum_path = os.path.join(out_dir, f"{label}_TTC_summary.csv")
    summary.to_csv(sum_path, index=False)
    print(f"Saved TTC summary:  {sum_path}\n")
    return summary


def run_cs3_ttc():
    if "CS3" not in ACTIVE_CASE_STUDIES:
        print("CS3 not enabled in CASE_STUDIES_TO_RUN — skipping TTC.")
        return
    cs3_labels = labels_for_case("CS3")
    if not cs3_labels:
        print("No CS3 files registered in FILE_META.")
        return
    pooled_by_label: dict[str, dict[str, pd.DataFrame]] = {}
    for label in cs3_labels:
        paths = discover_run_csv_paths(label)
        if not paths:
            folder = os.path.join(CASE_RUNS_DIR, label)
            leg = FILES_LEGACY.get(label, "")
            print(f"[SKIP] {label}: no CSVs -> {folder} / {leg}")
            continue
        print(f"Processing {label} for TTC ({len(paths)} run(s)): V2V-1D, V2V-2D, V2Ped, V2Bike ...")
        v2v_1d_parts, v2v_2d_parts, ped_parts, bike_parts = [], [], [], []
        saw_xy_theta = False
        for path in paths:
            df = load_csv_for_ttc(path)
            if df.empty:
                print(f"[{label}] run {path}: empty / unusable.")
                continue
            if {"x", "y", "theta"}.issubset(df.columns):
                saw_xy_theta = True
            v2v_1d_i = compute_ttc_v2v_1d(df)
            if v2v_1d_i.empty:
                print(f"    [{label}] V2V-1D (link) empty — diagnostic: {diagnose_ttc_v2v_1d(df)}")
            v2v_1d_parts.append(v2v_1d_i)
            v2v_2d_parts.append(compute_ttc_v2v_2d_intersection(df))
            ped_i, bike_i = compute_ttc_v2vru(df)
            if not ped_i.empty:
                ped_parts.append(ped_i)
            if not bike_i.empty:
                bike_parts.append(bike_i)
        if (v2v_1d_parts or v2v_2d_parts) and not saw_xy_theta:
            print(
                f"[{label}] x/y/theta not in CSV — V2P/V2B TTC skipped; add trajectory columns from sim export."
            )
        _v2v_cols = ["follower_id", "leader_id", "ttc", "follower_type"]
        v2v_1d = pd.concat(v2v_1d_parts, ignore_index=True) if v2v_1d_parts else pd.DataFrame(columns=_v2v_cols)
        v2v_2d = pd.concat(v2v_2d_parts, ignore_index=True) if v2v_2d_parts else pd.DataFrame(columns=_v2v_cols)
        ped = pd.concat(ped_parts, ignore_index=True) if ped_parts else pd.DataFrame(
            columns=["veh_id", "vru_id", "vru_class", "ttc", "veh_type"]
        )
        bike = pd.concat(bike_parts, ignore_index=True) if bike_parts else pd.DataFrame(
            columns=["veh_id", "vru_id", "vru_class", "ttc", "veh_type"]
        )
        if v2v_1d.empty and v2v_2d.empty and ped.empty and bike.empty:
            print(f"[{label}] no TTC data after pooling runs.")
            continue
        pool_note = f" — pooled over {len(paths)} run(s)" if len(paths) > 1 else ""
        print(f"Pooled distributions{pool_note} (counts are total unique pairs×runs).")
        pooled_by_label[label] = {
            "v2v_1d": v2v_1d, "v2v_2d": v2v_2d, "ped": ped, "bike": bike,
        }
        plot_ttc_distributions(v2v_1d, v2v_2d, ped, bike, label)

        mov_map = load_cs3_vehicle_movement_map()
        if not mov_map:
            print(f"[{label}] Part 2 skipped: no vehicle O-D map from {CS3_TRIPS_XML}")
        else:
            n_lt = sum(1 for m in mov_map.values() if m == "Left Turn")
            n_rt = sum(1 for m in mov_map.values() if m == "Right Turn")
            n_th = sum(1 for m in mov_map.values() if m == "Through")
            print(
                f"[{label}] Part 2 O-D map: {len(mov_map)} vehicles "
                f"(LT={n_lt}, RT={n_rt}, Through={n_th}) from {CS3_TRIPS_XML}"
            )
            summarize_ttc_by_movement_and_type(v2v_1d, v2v_2d, ped, bike, label, mov_map)

    plot_cs3_combined_fleet_cdfs(pooled_by_label)


if __name__ == "__main__":
    run_cs3_ttc()

CS3 not enabled in CASE_STUDIES_TO_RUN — skipping TTC.
